# Travel Matrix - Real Data Random Forest Predictor

Is notebook mein humne Python ki **Standard Basic Libraries** (`os`, `sys`, `math`, `json`, `datetime`, `random`, `collections`) ke saath-saath **Basic Data Science Libraries** (`matplotlib`, `seaborn`, `pandas`, `numpy`) ka istemaal kiya hai Data Analysis (EDA) aur Machine Learning ke liye.

In [ ]:
import os
import sys
import math
import json
import random
import collections
from datetime import datetime

# Basic Data Science & Visualization Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Basics
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

print("✅ All Basic Python & Data Science Libraries Imported Successfully!")

## 1. System Logging & Basic Module Usage
Standard `os` aur `datetime` library ka use project setup info print karne ke liye.

In [ ]:
current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Experiment Run Time: {current_time}")
print(f"Current Directory (OS): {os.getcwd()}")
print(f"Python Version (Sys): {sys.version.split(' ')[0]}")

## 2. Generating Real Indian Routes Dataset
Hum base data tayar karte hain aur custom probability & features model karte hain.

In [ ]:
real_routes = [
    {"source": "Delhi", "dest": "Jaipur", "dist": 280, "base_cost": 800, "mode": "Bus / Car"},
    {"source": "Delhi", "dest": "Agra", "dist": 230, "base_cost": 600, "mode": "Bus / Car"},
    {"source": "Mumbai", "dest": "Pune", "dist": 150, "base_cost": 500, "mode": "Bus / Car"},
    {"source": "Bangalore", "dest": "Mysore", "dist": 145, "base_cost": 400, "mode": "Bus / Car"},
    {"source": "Delhi", "dest": "Chandigarh", "dist": 250, "base_cost": 700, "mode": "Bus / Car"},
    
    {"source": "Mumbai", "dest": "Goa", "dist": 590, "base_cost": 2500, "mode": "Train"},
    {"source": "Bangalore", "dest": "Chennai", "dist": 350, "base_cost": 1200, "mode": "Train"},
    {"source": "Bangalore", "dest": "Hyderabad", "dist": 570, "base_cost": 1800, "mode": "Train"},
    {"source": "Delhi", "dest": "Manali", "dist": 530, "base_cost": 1500, "mode": "Train"},
    {"source": "Delhi", "dest": "Lucknow", "dist": 550, "base_cost": 1600, "mode": "Train"},
    {"source": "Mumbai", "dest": "Ahmedabad", "dist": 520, "base_cost": 1500, "mode": "Train"},
    
    {"source": "Delhi", "dest": "Mumbai", "dist": 1400, "base_cost": 6000, "mode": "Flight"},
    {"source": "Bangalore", "dest": "Delhi", "dist": 2100, "base_cost": 7500, "mode": "Flight"},
    {"source": "Chennai", "dest": "Delhi", "dist": 2200, "base_cost": 8000, "mode": "Flight"},
    {"source": "Kolkata", "dest": "Mumbai", "dist": 1900, "base_cost": 7000, "mode": "Flight"},
    {"source": "Hyderabad", "dest": "Delhi", "dist": 1600, "base_cost": 6500, "mode": "Flight"}
]

dataset = []
random.seed(42)
np.random.seed(42)

for route in real_routes:
    for _ in range(100):
        variation_dist = math.floor(route["dist"] * random.uniform(0.7, 1.3))
        
        if route["mode"] == "Flight":
            variation_budget = math.floor(route["base_cost"] * random.uniform(0.8, 3.0))
        elif route["mode"] == "Train":
            variation_budget = math.floor(route["base_cost"] * random.uniform(0.6, 2.5))
        else:
            variation_budget = math.floor(route["base_cost"] * random.uniform(0.5, 2.0))
            
        mode = route["mode"]
        
        # Introduce Overlap Noise for Realistic Accuracy
        if variation_dist > 1000 and variation_budget < 4500:
            mode = "Train"
        elif variation_dist < 400 and variation_budget > 3000:
            if random.random() < 0.3:
                mode = "Bus / Car"
        elif random.random() < 0.12: # Random noise factor
            mode = random.choice(["Bus / Car", "Train", "Flight"])

        dataset.append({
            "distance_km": variation_dist,
            "budget": variation_budget,
            "transport_mode": mode
        })

df = pd.DataFrame(dataset)
print(f"Database Volume: Generated {len(df)} records.")
display(df.head())


## 3. Exploratory Data Analysis (EDA) using `matplotlib` & `seaborn`
Data structure aur patterns ko deeply analyze karte hain model train karne se pehle.

In [ ]:
# Set style using seaborn
sns.set_theme(style="whitegrid")

# Using basic Collections library to count instances
counter = collections.Counter(df['transport_mode'])
print("Class Distribution Check (Collections Library):", dict(counter))

plt.figure(figsize=(10, 5))

# 1. Plotting class distributions
plt.subplot(1, 2, 1)
sns.countplot(data=df, x='transport_mode', palette='muted')
plt.title("Transport Mode Frequency")
plt.xlabel("Mode")
plt.ylabel("Count")

# 2. Plotting feature correlation (Scatter Plot)
plt.subplot(1, 2, 2)
sns.scatterplot(data=df, x='distance_km', y='budget', hue='transport_mode')
plt.title("Distance vs Budget per Mode")
plt.xlabel("Distance (km)")
plt.ylabel("Estimated Budget (₹)")

plt.tight_layout()
plt.show()

## 4. Modeling via RandomForest
Ab apne analytical findings par ek secure Machine Learning classifier array banayenge.

In [ ]:
X = df[['distance_km', 'budget']]
y = df['transport_mode']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)

predictions = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"🎯 Random Forest Accuracy Score: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, predictions))

# Confusion Matrix Visualization
cm = confusion_matrix(y_test, predictions, labels=rf_model.classes_)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=rf_model.classes_, yticklabels=rf_model.classes_, cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 5. Model Saving Details & `json` Extraction
Exporting configuration utilizing Python basic libraries.

In [ ]:
joblib.dump(rf_model, 'transport_model.pkl')
print(f"[✓] Model successfully serialized at: {os.path.abspath('transport_model.pkl')}")

model_metadata = {
    "created_at": datetime.now().isoformat(),
    "algorithm": "RandomForestClassifier",
    "accuracy": math.floor(accuracy * 100),
    "features_used": ["distance_km", "budget"],
    "target_classes": list(rf_model.classes_)
}

# Json library ka use karke metadata save kar rahe hain
with open('model_metadata.json', 'w') as f:
    json.dump(model_metadata, f, indent=4)

print("[✓] Metadata logic stored securely via JSON.")